# Load the  dataset from online

Load roughly 22GB dataset of [simulated financial transactions from Kaggle](https://www.kaggle.com/datasets/conorsully1/simulated-transactions)

In [31]:
import pynvml
pynvml.nvmlInit()
pynvml.nvmlDeviceGetName(pynvml.nvmlDeviceGetHandleByIndex(0))
mem = pynvml.nvmlDeviceGetMemoryInfo(pynvml.nvmlDeviceGetHandleByIndex(0))
mem = mem.total/1e9
if mem < 24:
    !wget https://storage.googleapis.com/rapidsai/polars-demo/transactions-t4-20.parquet -O transactions.parquet
else:
    !wget https://storage.googleapis.com/rapidsai/polars-demo/transactions.parquet -O transactions.parquet

--2025-05-28 21:06:28--  https://storage.googleapis.com/rapidsai/polars-demo/transactions.parquet
Resolving storage.googleapis.com (storage.googleapis.com)... 142.250.189.219, 142.250.189.251, 142.251.32.59, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|142.250.189.219|:443... connected.
200 OKequest sent, awaiting response... 
Length: 4274457161 (4.0G) [application/octet-stream]
Saving to: ‘transactions.parquet’

transactions.parque 100%[===================>]   3.98G  82.8MB/s    in 50s     

2025-05-28 21:07:19 (80.9 MB/s) - ‘transactions.parquet’ saved [4274457161/4274457161]



# Create the DuckDB database

In [4]:
import duckdb

# Paths
db_path = 'transactions.duckdb'
parquet_path = 'transactions.parquet'

# Connect to DuckDB
conn = duckdb.connect(db_path)

# Load Parquet directly into a table
conn.execute(f"""
CREATE OR REPLACE TABLE trxns AS 
SELECT * FROM read_parquet('{parquet_path}');
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

# Pull the data using GPU Polars

Pull the data from the DuckDB database 

In [5]:
import polars as pl

pl_trxns = pl.from_arrow(
    duckdb.connect("transactions.duckdb")
           .execute("SELECT * FROM trxns")
           .arrow()
).lazy()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Observe the first 5 rows of the data

In [6]:
(
    pl_trxns
    .head()
    .collect(engine = "gpu")
)

CUST_ID,START_DATE,END_DATE,TRANS_ID,DATE,YEAR,MONTH,DAY,EXP_TYPE,AMOUNT
str,date,date,str,date,i64,i64,i64,str,f64
"""CI6XLYUMQK""",2015-05-01,null,"""T8I9ZB5A6X90UG8""",2015-09-11,2015,9,11,"""Motor/Travel""",20.27
"""CI6XLYUMQK""",2015-05-01,null,"""TZ4JSLS7SC7FO9H""",2017-02-08,2017,2,8,"""Motor/Travel""",12.85
"""CI6XLYUMQK""",2015-05-01,null,"""TTUKRDDJ6B6F42H""",2015-08-01,2015,8,1,"""Housing""",383.8
"""CI6XLYUMQK""",2015-05-01,null,"""TDUHFRUKGPPI6HD""",2019-03-16,2019,3,16,"""Entertainment""",5.72
"""CI6XLYUMQK""",2015-05-01,null,"""T0JBZHBMSVRFMMD""",2015-05-15,2015,5,15,"""Entertainment""",11.06


# Filter and group transactions by year, month, and type, then compute total, average, and count of amounts, returning only high-value groups

In [7]:
%%time

filtered_df = (
    pl_trxns
    .filter(
        (pl.col("YEAR").is_in([2015, 2017, 2019])) &
        (pl.col("EXP_TYPE").is_in(["Motor/Travel", "Entertainment", "Housing"])) &
        (pl.col("AMOUNT") > 10)
    )
)

result = (
    filtered_df
    .group_by(["YEAR", "MONTH", "EXP_TYPE"])
    .agg([
        pl.len().alias("num_transcations"),
        pl.sum("AMOUNT").alias("total_amount"),
        pl.mean("AMOUNT").alias("avg_amount")
    ])
    .filter(pl.col("total_amount") > 50)
    .sort(["total_amount", "avg_amount"], descending = [True, True])
    .collect(engine = "gpu")
)


CPU times: user 2.36 s, sys: 1.02 s, total: 3.38 s
Wall time: 3.34 s


# Save the result back to DuckDB database

In [59]:
conn.execute("CREATE OR REPLACE TABLE result AS SELECT * FROM result")

# Close the databse connection

In [63]:
conn.close()